In [2]:
from preprocess import *

TRAIN_ROOT = "../data/NCT-CRC-HE-100K"
TEST_ROOT = "../data/CRC-VAL-HE-7K"

#will change every time --> alternate held_out class
HELD_OUT = "MUC"

In [3]:
train, val, known = make_splits(TRAIN_ROOT, HELD_OUT)
test = make_test_samples(TEST_ROOT, known, HELD_OUT)

print("all known classes:", known)

print("train size:", len(train))
print("validate size:", len(val))
print("test size:", len(test))

print(sum(l == -1 for _, l in test), "unknown test images")

all known classes: ['ADI', 'BACK', 'DEB', 'LYM', 'MUS', 'NORM', 'STR', 'TUM']
train size: 72886
validate size: 18218
test size: 7180
1035 unknown test images


In [4]:
train_loader = make_loader(train, train=True)
val_loader = make_loader(val, train=False)
test_loader = make_loader(test, train=False)

x, y = next(iter(train_loader))
print(x.shape, y[:10])

torch.Size([64, 3, 224, 224]) tensor([6, 5, 1, 5, 1, 0, 1, 7, 1, 2])


In [ ]:
import torch
from model import make_model, train_one_epoch, evaluate

#params
learning_rate = 0.0001
epochs = 5

device = "cuda" if torch.cuda.is_available() else "cpu"
#make one output per known class (8 since 1 is always excluded)
model = make_model(len(known)).to(device)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)


for epoch in range(epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
    print(f"epoch {epoch+1}: train loss {tr_loss:.3f} acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f}")

# x, y = next(iter(train_loader))

# #holds scores
# out = model(x.to(device))

# #cuda or gpu
# print(device)
# print(out.shape)